In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
data = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/train.csv')

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data['Heart Disease'].value_counts()

In [ ]:
data.drop('id',axis = 1,inplace = True)

In [ ]:
data["Heart Disease"] = data["Heart Disease"].map({"Absence": 0,"Presence": 1})

In [ ]:
data.dtypes

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
data.hist(figsize=(20, 15))
plt.show()

In [ ]:
data.corr()

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop("Heart Disease", axis=1)
y = data["Heart Disease"]

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,
    random_state=42, 
    stratify=y         
)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
rf = RandomForestClassifier(random_state=42,n_estimators=100,max_depth=10) 

In [ ]:
rf.fit(X_train, y_train)

In [ ]:
y_pred_proba = rf.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print("Test ROC-AUC:", test_auc)

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=10,               
    cv=5,                    
    scoring='roc_auc',       
    n_jobs=-1,
    random_state=42
)

In [ ]:
X_train.shape

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",   
    random_state=42,
    n_jobs=-1
)

param_dist = {
    "n_estimators": [300, 500, 700],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 1, 3]
}

In [ ]:
param_dist = {
    "n_estimators": [300, 500, 700],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 1, 3]
}

In [ ]:
random_searchx = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=20,        
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1
)

In [ ]:
random_searchx.fit(X_train, y_train)

print("Best Params:", random_searchx.best_params_)
print("Best AUC:", random_searchx.best_score_)

In [ ]:
best_xgb = random_searchx.best_estimator_
y_pred_probax = best_xgb.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print("Test ROC-AUC:", test_auc)

In [ ]:
test_data = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/test.csv')

In [ ]:
ids = test_data["id"]
X_test_model = test_data.drop(columns=["id"])

In [ ]:
test_data.columns

In [ ]:
y_pred = best_xgb.predict_proba(X_test_model)[:, 1]

In [ ]:
submission = pd.DataFrame({
    "id": ids,
    "Heart Disease": y_pred
})

submission.to_csv("submission.csv", index=False)